# Rotten Fruit Detection Dataset Prep - Colab

Use this notebook to validate the Roboflow object-detection dataset and generate model-agnostic artifacts. For SSD EfficientDet D5 with the TensorFlow Object Detection API, use the generated `*.record` files and `label_map.pbtxt`.

In [ ]:
import sys
from pathlib import Path

import tensorflow as tf

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("Working directory:", Path.cwd())

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/rotten-fruit-detection-ai.git"
PROJECT_DIR = Path("/content/rotten-fruit-detection-ai")

if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    %cd {PROJECT_DIR}
    !git pull

%cd {PROJECT_DIR}
print("Project dir:", PROJECT_DIR)

In [ ]:
%pip install -q -r requirements.txt

Expected zip contents:

```text
dataset/
  train/
    image1.jpg
    image2.jpg
    _annotations.csv
  valid/
    image3.jpg
    _annotations.csv
  test/
    image4.jpg
    _annotations.csv
```

In [ ]:
DRIVE_DATA_ZIP_PATH = Path("/content/drive/MyDrive/dataset.zip")
LOCAL_DATA_ZIP_PATH = PROJECT_DIR / "dataset.zip"
EXTRACT_DIR = PROJECT_DIR / "data"
LOCAL_DATA_DIR = EXTRACT_DIR / "dataset"
OUTPUT_DIR = Path("/content/drive/MyDrive/rotten-fruit-artifacts")

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Drive dataset zip:", DRIVE_DATA_ZIP_PATH)
print("Artifacts output:", OUTPUT_DIR)

In [ ]:
import shutil
import zipfile


def has_train_test(path):
    child_names = {child.name.lower() for child in Path(path).iterdir() if child.is_dir()}
    return "train" in child_names and "test" in child_names


def find_dataset_root(search_dir):
    search_dir = Path(search_dir)
    if has_train_test(search_dir):
        return search_dir

    for path in search_dir.rglob("*"):
        if path.is_dir() and has_train_test(path):
            return path

    raise ValueError(f"Could not find a folder with train and test inside {search_dir}.")


if not DRIVE_DATA_ZIP_PATH.exists():
    raise FileNotFoundError(f"Could not find dataset zip: {DRIVE_DATA_ZIP_PATH}")

shutil.copy2(DRIVE_DATA_ZIP_PATH, LOCAL_DATA_ZIP_PATH)

with zipfile.ZipFile(LOCAL_DATA_ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

LOCAL_DATA_DIR = find_dataset_root(EXTRACT_DIR)
print("Dataset root:", LOCAL_DATA_DIR)

In [ ]:
possible_src_dirs = [
    PROJECT_DIR / "src",
    Path.cwd() / "src",
    Path.cwd().parent / "src",
    Path("/content/rotten-fruit-detection-ai/src"),
]

SRC_DIR = next((path for path in possible_src_dirs if path.exists()), None)
if SRC_DIR is None:
    raise FileNotFoundError("Could not find src folder. Check PROJECT_DIR.")

sys.path.insert(0, str(SRC_DIR))
print("Using src folder:", SRC_DIR)

In [ ]:
from train import prepare_training_data

dataset, summary = prepare_training_data(
    data_dir=LOCAL_DATA_DIR,
    output_dir=OUTPUT_DIR,
    export_tensorflow_records=True,
)

summary

Generated artifacts include:

- `class_names.txt`
- `dataset_summary.json`
- `train_annotations.json`, `valid_annotations.json`, `test_annotations.json`
- `label_map.pbtxt`
- `train.record`, `valid.record`, `test.record`

For SSD EfficientDet D5, plug the `.record` paths and `label_map.pbtxt` into the TensorFlow Object Detection API pipeline config.